# Widget backpressure fixture

Matches the original bug report: `@interact`-decorated matplotlib plot driven by a FloatSlider. Rapid jagged back-and-forth slider drive floods the kernel with comm_msg updates faster than matplotlib can re-render. The visible slider position diverges from what the kernel has last processed (`kernel_slider_value` vs the DOM's `aria-valuenow`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

plt.style.use('dark_background')
x = np.linspace(0, 2 * np.pi, 200)

# Hoist the slider so the probe cell can read slider.value directly.
# `@interact` accepts a pre-built widget just the same.
slider = FloatSlider(min=0.5, max=5, step=0.1, value=1, description='Frequency')

@interact(freq=slider)
def plot(freq):
    plt.figure(figsize=(8, 4), facecolor='#1a1a1a')
    ax = plt.gca()
    ax.set_facecolor('#1a1a1a')
    plt.plot(x, np.sin(freq * x), color='#6366f1', linewidth=2)
    plt.ylim(-1.5, 1.5)
    plt.title(f'sin({freq:.1f}x)', color='white')
    plt.grid(True, alpha=0.3, color='gray')
    plt.tick_params(colors='white')
    for side in ('bottom', 'top', 'left', 'right'):
        ax.spines[side].set_color('gray')
    plt.show()

In [ ]:
# Round-trip through the kernel. If this lags the DOM's aria-valuenow
# after you've stopped moving, the comm_msg queue is draining — i.e.,
# backpressure. If it's permanently stuck, the sync or kernel is wedged.
print(f'kernel_slider_value={slider.value}')

In [ ]:
# Binary-blob parity: generate a PNG and display it. The daemon writes
# binary outputs to the blob store; the frontend resolves via the HTTP
# blob-server port exposed by get_blob_port. If this image renders under
# the Electron harness, the blob pipeline is intact.
from PIL import Image, ImageDraw
from IPython.display import Image as IPImage
import io

img = Image.new('RGB', (320, 80), color=(30, 36, 48))
draw = ImageDraw.Draw(img)
draw.rectangle([10, 10, 310, 70], outline=(180, 200, 220), width=2)
draw.text((20, 30), f'slider = {slider.value:.2f}', fill=(200, 220, 240))
buf = io.BytesIO()
img.save(buf, format='PNG')
IPImage(data=buf.getvalue(), format='png')